In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 44 — Level of service for car and transit at two levels: the 781 TAZs and the 25 V2 areas of the LRT corridor (`LOS_skims_TAZ_and_V2.ipynb`)

The chain's level of service (the skims) lives on the **25 V2 areas**: step 26 (§6x) fills the
components mode by mode, step 31 (§6ac) assembles one complete skim per mode, and the capture
runs on them. Every component is defined per area pair and, where it comes from a network, is
routed between one **representative point** per area — so the walk, the first and last mile
and the path are those of the area's point, not of its TAZs. This step builds the same
level of service **at TAZ level**, for all 781 study TAZs, by routing every TAZ pair on the
networks the chain already uses, and then **aggregates the TAZ skims to the 25 areas** with
population (origin) and employment (destination) weights, to put beside the area skims of
step 31 — the same quantities from the bottom up.

**Definitions, as in the chain.** Generalized cost in minutes,
`GC = IVT + 2·walk + 2·wait + 8·transfers` (money out of the comparison by decision); walking
at 4.8 km/h on the straight line × 1.3; wait = half the headway, capped at 10 minutes.

- **Car.** In-vehicle time on the fastest path at the May 2026 07:00 car speeds (step 37's
  network: the bus street network's topology with the car speeds transferred to its segments,
  30 km/h assumed on the segments without a speed), between the **TAZ polygon centroids**,
  with a straight-line connector at 20 km/h to the nearest network node at each end, plus a
  **3-minute terminal allowance** (parking and walking at both ends). No walk, wait or transfer
  term. The chain's default car skim is instead the 2017/18 survey's door-to-door time
  between areas (step 26); step 37 showed the two agree to within 10 % on the trunk.
- **Transit.** The scheduled bus and Metronit services of the national GTFS on the
  representative Tuesday (2 June 2026, step 29) departing in 06:00–09:00, as a
  **frequency-based line graph**: a node per stop and a node per (line, direction, stop);
  *ride* edges between a line's consecutive stops timed at step 30's observed running time
  where it exists (the trips routed over the May 2026 bus link speeds, dwell included) and
  at the schedule otherwise; *boarding* edges from the stop to the line costed at 2 × the wait
  (half the line-direction's 07:00–08:00 headway at that stop, capped at 10 min) plus the
  8-minute transfer penalty; *alighting* free; *walk transfers* between stops within 300 m;
  *access and egress* from the TAZ centroid to every stop within 1,000 m (and to the nearest
  served stop in any case). The cheapest path in generalized cost is found per TAZ pair,
  and one transfer penalty is refunded (the first boarding is not a transfer). This is the
  **best single path**: its wait is the chosen line's own headway, which is the chain's
  `BUS_WAIT_RULE = 'best_line'` reading (§6x addendum 6) rather than the default pooled
  headway of all direct lines on the pair — the comparison below uses both area-level
  variants. The Metronit is part of the graph; the path records whether it rides it.

The planned LRT is not in this step (its TAZ-level skim is task C3 / step 39).

In [2]:
import time, gzip
import numpy as np
import pandas as pd
import geopandas as gpd
import shapefile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from scipy.spatial import cKDTree

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/los'; FIG = 'Output/figures'; os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
W_WALK, W_WAIT, TRANSFER_PEN, WAIT_CAP = 2.0, 2.0, 8.0, 10.0      # the chain's generalized-cost parameters (docs/LRT_CAPTURE_PLAN.md, step 26)
WALK_SPEED, DETOUR = 80.0, 1.3                                     # m/min, straight-line → walking-network factor
ACCESS_M, TRANSFER_M = 1000.0, 300.0                               # straight-line radii: TAZ centroid → stops; stop → stop walk transfer
PEAK = (7 * 3600, 8 * 3600); AM_HOURS = 3.0
CAR_FALLBACK_KMH, CAR_CONNECTOR_KMH, CAR_TERMINAL_MIN, MATCH_TOL, HOUR = 30.0, 20.0, 3.0, 3.0, 7   # step 37's car-routing assumptions
TRUNK = list(range(201, 211))

# TAZs: polygon centroids, population and employment, V2 membership
tzg = gpd.read_file('Input/TAZ_North/TAZ_North.shp', engine='pyogrio')
TAZ = pd.DataFrame({'TAZ': tzg['TAZ_NUMBER'].astype(int), 'cx': tzg.geometry.centroid.x, 'cy': tzg.geometry.centroid.y}).drop_duplicates('TAZ').set_index('TAZ').sort_index()
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID')
TAZ['pop'] = zon['POPULATION'].reindex(TAZ.index).fillna(0); TAZ['emp'] = zon['EMPL_TOT'].reindex(TAZ.index).fillna(0)
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
AREAS = list(areas.index); names = areas['AggAreaName']; A = len(AREAS)
TAZ['area'] = key.set_index('TAZ')['AggCode'].reindex(TAZ.index)
TAZS = list(TAZ.index); NT = len(TAZS); tpos = {t: i for i, t in enumerate(TAZS)}
print(f'{NT} TAZs ({int(TAZ["area"].notna().sum())} in the 25 V2 areas); population {TAZ["pop"].sum():,.0f}, employment {TAZ["emp"].sum():,.0f}')

def path_sums(pred, mats, N):
    """Sum of each matrix's edge values along the predecessor tree, for every node, by pointer jumping (vectorised)."""
    valid = pred >= 0; idx = np.where(valid)[0]
    acc = {}
    for k, M in mats.items():
        e = np.zeros(N); e[idx] = np.asarray(M[pred[idx], idx]).ravel(); acc[k] = e
    jump = np.where(valid, pred, np.arange(N))
    for _ in range(40):
        for k in acc: acc[k] = acc[k] + acc[k][jump]
        nj = jump[jump]
        if np.array_equal(nj, jump): break
        jump = nj
    return acc

def skim_from_sources(G, mats, src_nodes, dst_nodes, chunk=40):
    """Dijkstra from every source node; returns dict of (len(src) × len(dst)) arrays: 'gc' and one per matrix in mats."""
    N = G.shape[0]; res = {'gc': np.full((len(src_nodes), len(dst_nodes)), np.nan)}; res.update({k: np.full_like(res['gc'], np.nan) for k in mats})
    for s0 in range(0, len(src_nodes), chunk):
        srcs = src_nodes[s0:s0 + chunk]
        D, P = dijkstra(G, indices=srcs, directed=True, return_predecessors=True)
        for i, s in enumerate(srcs):
            res['gc'][s0 + i] = D[i, dst_nodes]
            sums = path_sums(P[i], mats, N)
            for k in mats: res[k][s0 + i] = sums[k][dst_nodes]
    for k in res: res[k][~np.isfinite(res['gc'])] = np.nan
    return res

781 TAZs (174 in the 25 V2 areas); population 2,556,484, employment 953,662


## 1. Car level of service between the 781 TAZ centroids

In [3]:
t0 = time.time()
car = gpd.read_file('Output/car_speed/GoogleSpeed_202605_North.gpkg', engine='pyogrio')
for d in ('AB', 'BA'): car[f'v7_{d}'] = car[f'SPD_{HOUR}_{d}'].fillna(0)
BB = tzg.total_bounds; rd = shapefile.Reader('Input/BusSpeedData/Streets/Streets'); segs = []
for s, rec in zip(rd.iterShapes(), rd.iterRecords()):
    b = s.bbox
    if not (b[2] > BB[0] and b[0] < BB[2] and b[3] > BB[1] and b[1] < BB[3]): continue
    segs.append((LineString(s.points), rec['DIR']))
tree = STRtree(car.geometry.values); G_car, L_car = car.geometry.values, car.geometry.length.values
match = np.full(len(segs), -1); same = np.ones(len(segs), bool)
for k, (L, dr) in enumerate(segs):
    a, b = Point(L.coords[0]), Point(L.coords[-1])
    for c in tree.query(L.buffer(MATCH_TOL)):
        g = G_car[c]
        if g.distance(a) < MATCH_TOL and g.distance(b) < MATCH_TOL and L.length <= L_car[c] + MATCH_TOL:
            match[k] = c; same[k] = g.project(a) < g.project(b); break
m = match >= 0; idx = np.where(m, match, 0); ab, ba = car['v7_AB'].values[idx], car['v7_BA'].values[idx]
e = pd.DataFrame({'ax': [s[0].coords[0][0] for s in segs], 'ay': [s[0].coords[0][1] for s in segs], 'bx': [s[0].coords[-1][0] for s in segs], 'by': [s[0].coords[-1][1] for s in segs],
                  'len': [s[0].length for s in segs], 'dir': [s[1] for s in segs], 'vAB': np.where(m, np.where(same, ab, ba), 0.0), 'vBA': np.where(m, np.where(same, ba, ab), 0.0)})
nodes = {}
def node(x, y):
    k = (round(x), round(y))
    if k not in nodes: nodes[k] = len(nodes)
    return nodes[k]
e['a'] = [node(x, y) for x, y in zip(e['ax'], e['ay'])]; e['b'] = [node(x, y) for x, y in zip(e['bx'], e['by'])]
NC = len(nodes); XY = np.array([k for k, v in sorted(nodes.items(), key=lambda kv: kv[1])], dtype=float)
rows, cols, tmin, lens, fbl = [], [], [], [], []
for a, b, ln, dr, vab, vba in zip(e['a'], e['b'], e['len'], e['dir'], e['vAB'], e['vBA']):
    for (u, w, v, open_) in ((a, b, vab, dr in (0, 1)), (b, a, vba, dr in (0, -1))):
        if not open_: continue
        speed, f = (v, 0.0) if v > 0 else (CAR_FALLBACK_KMH, ln)
        rows.append(u); cols.append(w); tmin.append(ln / 1000 / speed * 60); lens.append(ln); fbl.append(f)
Gc = coo_matrix((tmin, (rows, cols)), shape=(NC, NC)).tocsr(); GLc = coo_matrix((lens, (rows, cols)), shape=(NC, NC)).tocsr()
ncomp, lab = connected_components(Gc, directed=True, connection='strong'); giant = np.bincount(lab).argmax(); served = np.array(sorted(v for v in set(rows) | set(cols) if lab[v] == giant))
conn_d, conn_i = cKDTree(XY[served]).query(TAZ[['cx', 'cy']].values); anchor = served[conn_i]
print(f'car graph: {len(e):,} segments, {NC:,} nodes, giant component {len(served):,}; TAZ connectors {conn_d.min():.0f}–{conn_d.max():.0f} m (median {np.median(conn_d):.0f}); {time.time() - t0:.0f} s')
t0 = time.time(); R = skim_from_sources(Gc, {'km': GLc}, anchor, anchor)
car_t = R['gc'] + (conn_d[:, None] + conn_d[None, :]) / 1000 / CAR_CONNECTOR_KMH * 60; car_km = R['km'] / 1000 + (conn_d[:, None] + conn_d[None, :]) / 1000
np.fill_diagonal(car_t, np.nan); np.fill_diagonal(car_km, np.nan)
car_gc = car_t + CAR_TERMINAL_MIN
print(f'car TAZ skim: {np.isfinite(car_t).sum():,} of {NT * (NT - 1):,} pairs reachable; kerb-to-kerb time median {np.nanmedian(car_t):.1f} min (p10 {np.nanquantile(car_t, .1):.1f}, p90 {np.nanquantile(car_t, .9):.1f}); {time.time() - t0:.0f} s')
car_long = pd.DataFrame({'o': np.repeat(TAZS, NT), 'd': np.tile(TAZS, NT), 'car_ivt_min': car_t.ravel(), 'car_km': car_km.ravel(), 'car_gc_min': car_gc.ravel()}).dropna(subset=['car_ivt_min'])
car_long.to_csv(f'{OUT}/car_los_taz.csv.gz', index=False, float_format='%.2f', compression='gzip')

car graph: 54,507 segments, 48,898 nodes, giant component 47,609; TAZ connectors 6–3122 m (median 148); 14 s


car TAZ skim: 609,180 of 609,180 pairs reachable; kerb-to-kerb time median 48.3 min (p10 18.4, p90 80.8); 7 s


## 2. Transit level of service between the 781 TAZ centroids — the frequency-based line graph

In [4]:
t0 = time.time()
st = pd.read_csv('Output/gtfs/stop_times_study_area_am_trips.csv.gz'); stops = pd.read_csv('Output/gtfs/stops_study_area.csv').set_index('stop_id')
st['line'] = st['route_code'].astype(int).astype(str) + '_' + st['direction_id'].astype(int).astype(str)
st = st.sort_values(['trip_id', 'stop_sequence']); g = st.groupby('trip_id')
st['next_stop'] = g['stop_id'].shift(-1); st['next_arr'] = g['arr'].shift(-1); st['dt_min'] = (st['next_arr'] - st['dep']) / 60
seg_obs = pd.read_csv('Output/gtfs/bus_segments_observed.csv'); seg_obs = seg_obs[seg_obs['flag'].isin(['ok', 'ok_undirected']) & (seg_obs['observed_min'] > 0)].set_index(['stop_a', 'stop_b'])['observed_min']
# ride edges: per (line, stop_a, stop_b) the median scheduled time over the AM trips (peak trips preferred), replaced by step 30's observed time where it exists
rides = st.dropna(subset=['next_stop']).copy(); rides['next_stop'] = rides['next_stop'].astype(int)
rides = rides[rides['dt_min'] > 0]
rt = rides.groupby(['line', 'stop_id', 'next_stop']).agg(sched_min=('dt_min', 'median'), n_trips=('trip_id', 'nunique'), is_brt=('is_brt', 'max')).reset_index()
rt['obs_min'] = [seg_obs.get((a, b), np.nan) for a, b in zip(rt['stop_id'], rt['next_stop'])]
rt['ride_min'] = rt['obs_min'].where(rt['obs_min'].notna(), rt['sched_min'])
# boarding: per (line, stop) the peak-hour departures → headway → wait; AM departures / 3 where no peak departure
dep = st.groupby(['line', 'stop_id']).agg(n_peak=('peak', 'sum'), n_am=('trip_id', 'nunique')).reset_index()
dep['per_hour'] = np.where(dep['n_peak'] > 0, dep['n_peak'], dep['n_am'] / AM_HOURS); dep['wait_min'] = np.minimum(60 / dep['per_hour'] / 2, WAIT_CAP)
# nodes: stop nodes, line nodes, origin-TAZ and destination-TAZ nodes
STOPS = sorted(set(rt['stop_id']) | set(rt['next_stop'])); spos = {s: i for i, s in enumerate(STOPS)}; NS = len(STOPS)
ln_keys = sorted(set(zip(rt['line'], rt['stop_id'])) | set(zip(rt['line'], rt['next_stop']))); lpos = {k: NS + i for i, k in enumerate(ln_keys)}; NL = len(ln_keys)
O0 = NS + NL; D0 = O0 + NT; N = D0 + NT
is_brt_line = rt.groupby('line')['is_brt'].max().to_dict()
E = {'u': [], 'v': [], 'gc': [], 'ivt': [], 'walk': [], 'wait': [], 'board': [], 'brt': []}
def add_edge(u, v, gc, ivt=0.0, walk=0.0, wait=0.0, board=0.0, brt=0.0):
    E['u'].append(u); E['v'].append(v); E['gc'].append(gc); E['ivt'].append(ivt); E['walk'].append(walk); E['wait'].append(wait); E['board'].append(board); E['brt'].append(brt)
for r in rt.itertuples():                                                      # ride
    add_edge(lpos[(r.line, r.stop_id)], lpos[(r.line, r.next_stop)], r.ride_min, ivt=r.ride_min, brt=r.ride_min if r.is_brt else 0.0)
wait_of = dep.set_index(['line', 'stop_id'])['wait_min']
for (line, s), ln in lpos.items():                                            # board (with the transfer penalty, one refunded below) and alight
    w = wait_of.get((line, s), WAIT_CAP)
    add_edge(spos[s], ln, W_WAIT * w + TRANSFER_PEN, wait=w, board=1.0); add_edge(ln, spos[s], 0.0)
sxy = stops.loc[STOPS, ['x', 'y']].values; kd = cKDTree(sxy)
for i, j in kd.query_pairs(TRANSFER_M):                                        # walk transfers, both ways
    wk = np.hypot(*(sxy[i] - sxy[j])) * DETOUR / WALK_SPEED
    add_edge(i, j, W_WALK * wk, walk=wk); add_edge(j, i, W_WALK * wk, walk=wk)
n_acc = []
for ti, t in enumerate(TAZS):                                                  # access / egress: stops within ACCESS_M, and the nearest in any case
    c = TAZ.loc[t, ['cx', 'cy']].values.astype(float); near = kd.query_ball_point(c, ACCESS_M)
    if not near: near = [kd.query(c)[1]]
    n_acc.append(len(near))
    for i in near:
        wk = np.hypot(*(sxy[i] - c)) * DETOUR / WALK_SPEED
        add_edge(O0 + ti, i, W_WALK * wk, walk=wk); add_edge(i, D0 + ti, W_WALK * wk, walk=wk)
Gt = csr_matrix((E['gc'], (E['u'], E['v'])), shape=(N, N)); Gt.data = np.maximum(Gt.data, 1e-6)   # zero-cost alight edges kept as edges
mats_t = {k: csr_matrix((E[k], (E['u'], E['v'])), shape=(N, N)) for k in ('ivt', 'walk', 'wait', 'board', 'brt')}
print(f'transit graph: {NS:,} stops, {len(set(rt["line"])):,} line-directions, {NL:,} line nodes, {len(rt):,} ride edges ({rt["obs_min"].notna().mean():.0%} with step-30 observed times), '
      f'{len(E["u"]):,} edges in all; TAZ access stops within {ACCESS_M:.0f} m: median {np.median(n_acc):.0f}, {sum(1 for n in n_acc if n == 1 and True)} TAZs on the nearest stop only; {time.time() - t0:.0f} s')
t0 = time.time(); Rt = skim_from_sources(Gt, mats_t, np.arange(O0, O0 + NT), np.arange(D0, D0 + NT))
tr_gc = Rt['gc'] - TRANSFER_PEN * (Rt['board'] >= 1); tr_tr = np.maximum(Rt['board'] - 1, 0)
np.fill_diagonal(tr_gc, np.nan)
for k in Rt: np.fill_diagonal(Rt[k], np.nan)
reach = np.isfinite(tr_gc); walk_only = reach & (Rt['board'] == 0)
print(f'transit TAZ skim: {reach.sum():,} of {NT * (NT - 1):,} pairs reachable ({walk_only.sum():,} by walking alone); GC median {np.nanmedian(tr_gc):.1f} min (p10 {np.nanquantile(tr_gc, .1):.1f}, p90 {np.nanquantile(tr_gc, .9):.1f}); '
      f'transfers: {np.nanmean(tr_tr[reach & ~walk_only]):.2f} on average, {(tr_tr[reach & ~walk_only] == 0).mean():.0%} direct; paths riding the Metronit {(Rt["brt"][reach] > 0).mean():.0%}; {time.time() - t0:.0f} s')
tr_long = pd.DataFrame({'o': np.repeat(TAZS, NT), 'd': np.tile(TAZS, NT), 'transit_gc_min': tr_gc.ravel(), 'ivt_min': Rt['ivt'].ravel(), 'walk_min': Rt['walk'].ravel(), 'wait_min': Rt['wait'].ravel(),
                        'transfers': tr_tr.ravel(), 'boardings': Rt['board'].ravel(), 'brt_ivt_min': Rt['brt'].ravel()}).dropna(subset=['transit_gc_min'])
tr_long['walk_only'] = tr_long['boardings'] == 0
tr_long.to_csv(f'{OUT}/transit_los_taz.csv.gz', index=False, float_format='%.2f', compression='gzip')
# per-TAZ accessibility summary: median GC to the other V2-area TAZs, car and transit, and the transit / car ratio
v2 = np.array([pd.notna(TAZ.loc[t, 'area']) for t in TAZS])
acc = pd.DataFrame({'TAZ': TAZS, 'area': TAZ['area'].values, 'pop': TAZ['pop'].values, 'emp': TAZ['emp'].values,
                    'car_gc_median_to_v2_taz': np.nanmedian(np.where(v2[None, :], car_gc, np.nan), axis=1), 'transit_gc_median_to_v2_taz': np.nanmedian(np.where(v2[None, :], tr_gc, np.nan), axis=1),
                    'transit_reach_v2_taz_n': (reach & v2[None, :]).sum(axis=1), 'transit_direct_share_v2': np.nanmean(np.where(reach & v2[None, :] & ~walk_only, tr_tr == 0, np.nan), axis=1)})
acc['transit_over_car'] = acc['transit_gc_median_to_v2_taz'] / acc['car_gc_median_to_v2_taz']
acc.to_csv(f'{OUT}/los_taz_accessibility.csv', index=False, float_format='%.2f')

transit graph: 10,997 stops, 1,410 line-directions, 46,263 line nodes, 44,726 ride edges (97% with step-30 observed times), 263,778 edges in all; TAZ access stops within 1000 m: median 32, 92 TAZs on the nearest stop only; 3 s


transit TAZ skim: 608,400 of 609,180 pairs reachable (5,903 by walking alone); GC median 149.8 min (p10 74.1, p90 231.9); transfers: 1.37 on average, 17% direct; paths riding the Metronit 22%; 26 s


## 3. From the TAZs to the 25 V2 areas, and against the chain's area skims

Each area pair's level of service is the average of its TAZ pairs' values, weighted by the
origin TAZ's population × the destination TAZ's employment (the allocation rule of the chain;
equal weights where an area has neither, the Tsomet Kiryat Ata junction area). Unreachable
TAZ pairs are left out of the average and counted. The result is put beside step 31's area
skims — car (survey door-to-door), bus and Metronit (pooled headway, default) and the
`bus_wait_best_line` variant (the single best line's headway, closest to this step's best
path) — on the 90 trunk pairs weighted by the 2022 car trips (car) and transit trips
(transit), and over all 600 pairs.

In [5]:
tz_area = TAZ['area']; in_v2 = tz_area.notna().values
wgt_o = TAZ['pop'].values; wgt_d = TAZ['emp'].values
def aggregate(M):
    out = pd.DataFrame(np.nan, index=AREAS, columns=AREAS); cnt = pd.DataFrame(0, index=AREAS, columns=AREAS)
    for ao in AREAS:
        io = np.where(tz_area.values == ao)[0]
        for ad in AREAS:
            jd = np.where(tz_area.values == ad)[0]
            sub = M[np.ix_(io, jd)]; w = np.outer(wgt_o[io], wgt_d[jd]).astype(float)
            if ao == ad: np.fill_diagonal(sub, np.nan) if sub.shape[0] == sub.shape[1] else None
            ok = np.isfinite(sub)
            if ok.sum() == 0: continue
            if w[ok].sum() == 0: w = np.ones_like(w)
            out.loc[ao, ad] = np.average(sub[ok], weights=w[ok]); cnt.loc[ao, ad] = int(ok.sum())
    return out, cnt
AGG = {}
for k, M in [('car_ivt', car_t), ('car_gc', car_gc), ('car_km', car_km), ('transit_gc', tr_gc), ('transit_ivt', Rt['ivt']), ('transit_walk', Rt['walk']), ('transit_wait', Rt['wait']), ('transit_transfers', tr_tr), ('transit_brt_ivt', Rt['brt'])]:
    AGG[k], cnt = aggregate(M); AGG[k].to_csv(f'{OUT}/{k}_area_v2.csv', float_format='%.2f')
cnt.to_csv(f'{OUT}/taz_pairs_reachable_area_v2.csv')
AGG['transit_direct_share'], _ = aggregate((tr_tr == 0).astype(float) * np.where(reach & ~walk_only, 1.0, np.nan)); AGG['transit_direct_share'].to_csv(f'{OUT}/transit_direct_share_area_v2.csv', float_format='%.3f')
def rd31(path):
    m = pd.read_csv(path, index_col=0); m.columns = m.columns.astype(int); return m.reindex(index=AREAS, columns=AREAS).astype(float)
S31 = {'car_gc': rd31('Output/skims/skim_car_gc.csv'), 'car_ivt': rd31('Output/skims/skim_car_ivt.csv'), 'bus_gc': rd31('Output/skims/skim_bus_gc.csv'), 'bus_ivt': rd31('Output/skims/skim_bus_ivt.csv'),
       'bus_walk': rd31('Output/skims/skim_bus_walk.csv'), 'bus_wait': rd31('Output/skims/skim_bus_wait.csv'), 'bus_transfers': rd31('Output/skims/skim_bus_transfers.csv'), 'brt_gc': rd31('Output/skims/skim_brt_gc.csv'),
       'bus_gc_best_line': rd31('Output/skims/bus_wait_best_line/skim_bus_gc.csv'), 'bus_wait_best_line': rd31('Output/skims/bus_wait_best_line/skim_bus_wait.csv')}
S31['transit_best_gc'] = S31['bus_gc'].where(S31['brt_gc'].isna() | (S31['bus_gc'] <= S31['brt_gc']), S31['brt_gc'])
S31['car_gc_2026'] = rd31('Output/gc/car_ivt_network_area_v2.csv') + CAR_TERMINAL_MIN; np.fill_diagonal(S31['car_gc_2026'].values, np.nan)
flows = pd.read_csv('Output/skims/pair_flows_and_skims_2022.csv')
Wc = flows.pivot(index='o', columns='d', values='car_2022').reindex(index=AREAS, columns=AREAS).fillna(0); Wt = flows.pivot(index='o', columns='d', values='transit_2022').reindex(index=AREAS, columns=AREAS).fillna(0)
off = ~np.eye(A, dtype=bool); tk = np.array([[o in TRUNK and d in TRUNK for d in AREAS] for o in AREAS]) & off
def wm(M, W, mask): v, w = M.values, W.values * mask; ok = np.isfinite(v) & (w > 0); return np.average(v[ok], weights=w[ok]) if ok.any() else np.nan
rows = []
for label, M, W in [('car GC — TAZ-aggregated (2026 network + 3 min)', AGG['car_gc'], Wc), ('car GC — step 37 area points (2026 network + 3 min)', S31['car_gc_2026'], Wc), ('car GC — step 31 (survey door-to-door)', S31['car_gc'], Wc),
                    ('car IVT — TAZ-aggregated kerb to kerb', AGG['car_ivt'], Wc), ('car km — TAZ-aggregated path', AGG['car_km'], Wc),
                    ('transit GC — TAZ-aggregated best path (bus + Metronit)', AGG['transit_gc'], Wt), ('transit GC — step 31 cheaper of bus / Metronit (pooled wait)', S31['transit_best_gc'], Wt),
                    ('bus GC — step 31 (pooled wait)', S31['bus_gc'], Wt), ('bus GC — step 31 best-line wait variant', S31['bus_gc_best_line'], Wt),
                    ('transit IVT — TAZ-aggregated', AGG['transit_ivt'], Wt), ('bus IVT — step 31', S31['bus_ivt'], Wt), ('transit walk — TAZ-aggregated', AGG['transit_walk'], Wt), ('bus walk — step 31', S31['bus_walk'], Wt),
                    ('transit wait — TAZ-aggregated (best line)', AGG['transit_wait'], Wt), ('bus wait — step 31 pooled', S31['bus_wait'], Wt), ('bus wait — step 31 best line', S31['bus_wait_best_line'], Wt),
                    ('transit transfers — TAZ-aggregated', AGG['transit_transfers'], Wt), ('bus transfers — step 31 (0 direct / 1 assumed)', S31['bus_transfers'], Wt)]:
    rows.append({'quantity': label, 'trunk pairs (trip-weighted)': wm(M, W, tk), 'all pairs (trip-weighted)': wm(M, W, off), 'all pairs (simple mean)': np.nanmean(M.values[off])})
cmp_ = pd.DataFrame(rows); cmp_.to_csv(f'{OUT}/los_area_v2_vs_step31.csv', index=False, float_format='%.2f'); print(cmp_.round(1).to_string(index=False))
pairs = pd.DataFrame({'o': np.repeat(AREAS, A), 'd': np.tile(AREAS, A)}); pairs = pairs[pairs['o'] != pairs['d']].copy()
for k, M in [('car_gc_taz_agg', AGG['car_gc']), ('car_gc_step31_survey', S31['car_gc']), ('car_gc_step37_points', S31['car_gc_2026']), ('transit_gc_taz_agg', AGG['transit_gc']), ('transit_gc_step31_pooled', S31['transit_best_gc']), ('bus_gc_step31_best_line', S31['bus_gc_best_line']),
             ('transit_ivt_taz_agg', AGG['transit_ivt']), ('bus_ivt_step31', S31['bus_ivt']), ('transit_walk_taz_agg', AGG['transit_walk']), ('bus_walk_step31', S31['bus_walk']), ('transit_wait_taz_agg', AGG['transit_wait']), ('bus_wait_step31', S31['bus_wait']),
             ('transit_transfers_taz_agg', AGG['transit_transfers']), ('transit_direct_share_taz_agg', AGG['transit_direct_share']), ('taz_pairs', cnt)]:
    pairs[k] = [M.loc[o, d] for o, d in zip(pairs['o'], pairs['d'])]
pairs.insert(2, 'o_name', pairs['o'].map(names)); pairs.insert(3, 'd_name', pairs['d'].map(names)); pairs['car_2022'] = [Wc.loc[o, d] for o, d in zip(pairs['o'], pairs['d'])]; pairs['transit_2022'] = [Wt.loc[o, d] for o, d in zip(pairs['o'], pairs['d'])]
pairs.to_csv(f'{OUT}/los_area_v2_pairs.csv', index=False, float_format='%.2f')

                                                    quantity  trunk pairs (trip-weighted)  all pairs (trip-weighted)  all pairs (simple mean)
              car GC — TAZ-aggregated (2026 network + 3 min)                         13.3                       16.6                     25.1
         car GC — step 37 area points (2026 network + 3 min)                         13.0                       16.1                     24.3
                      car GC — step 31 (survey door-to-door)                         14.4                       16.9                     25.9
                       car IVT — TAZ-aggregated kerb to kerb                         10.3                       13.6                     22.1
                                car km — TAZ-aggregated path                          6.3                        8.9                     16.1
      transit GC — TAZ-aggregated best path (bus + Metronit)                         45.2                       54.2                     74.6
transi

In [6]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
def sc(a, x, y, w, title, xl_, yl_):
    ok = np.isfinite(x) & np.isfinite(y); a.scatter(x[ok], y[ok], s=6 + w[ok] / 40, color=BLUE, alpha=.55); lim = np.nanmax([x[ok].max(), y[ok].max()]) * 1.05
    a.plot([0, lim], [0, lim], color=INK2, lw=.8, ls='--'); a.set_xlim(0, lim); a.set_ylim(0, lim); a.set_title(title, fontsize=10); a.set_xlabel(xl_); a.set_ylabel(yl_); a.grid(color=GRID, lw=.6)
    for sp in ('top', 'right'): a.spines[sp].set_visible(False)
sc(ax[0], pairs['car_gc_step31_survey'].values, pairs['car_gc_taz_agg'].values, pairs['car_2022'].values, 'Car: TAZ-aggregated 2026 network vs step 31 survey', 'step 31 survey door-to-door (min)', 'TAZ-aggregated, 2026 network + 3 min (min)')
sc(ax[1], pairs['transit_gc_step31_pooled'].values, pairs['transit_gc_taz_agg'].values, pairs['transit_2022'].values, 'Transit GC: TAZ-aggregated best path vs step 31 (pooled wait)', 'step 31 cheaper of bus / Metronit (gen. min)', 'TAZ-aggregated best path (gen. min)')
sc(ax[2], pairs['bus_gc_step31_best_line'].values, pairs['transit_gc_taz_agg'].values, pairs['transit_2022'].values, 'Transit GC: TAZ-aggregated vs step 31 best-line wait', 'step 31 bus, best-line wait (gen. min)', 'TAZ-aggregated best path (gen. min)')
plt.tight_layout(); plt.savefig(f'{FIG}/los_area_v2_taz_agg_vs_step31.png', dpi=150); plt.close()
# maps: generalized cost from every TAZ to the busiest-employment trunk TAZ, car and transit
dest = int(TAZ[TAZ['area'].isin(TRUNK)]['emp'].idxmax()); jd = tpos[dest]
fig, ax = plt.subplots(1, 2, figsize=(14, 8))
for a, M, title, vmax in [(ax[0], car_gc[:, jd], f'Car GC to TAZ {dest} ({names[TAZ.loc[dest, "area"]]}), 07:00 network + 3 min', 60), (ax[1], tr_gc[:, jd], f'Transit GC to TAZ {dest}, best path 06:00–09:00 services', 120)]:
    tzg2 = tzg.assign(v=[M[tpos[int(t)]] if int(t) in tpos else np.nan for t in tzg['TAZ_NUMBER']])
    tzg2.plot(ax=a, column='v', cmap='RdYlGn_r', vmin=0, vmax=vmax, edgecolor=AXIS, linewidth=.2, legend=True, missing_kwds={'color': '#f0efec'}, legend_kwds={'shrink': .5, 'label': 'generalized minutes'})
    a.set_title(title, fontsize=10); a.set_axis_off()
plt.tight_layout(); plt.savefig(f'{FIG}/los_taz_gc_maps.png', dpi=150); plt.close(); print('figures written; map destination TAZ', dest)

figures written; map destination TAZ 1517


## 4. Results, outputs, limits

**Car, TAZ level.** 48,898 nodes, connectors 6–3,122 m (median 148); all 609,180 TAZ pairs
reachable; kerb-to-kerb time median 48 min over the whole study area (p10 18, p90 81), and
26 min (18 km) between the 174 corridor TAZs. Aggregated to the areas the car GC is **13.3
min on the trunk pairs** (trip-weighted) against step 37's 13.0 between the area points and
the survey's 14.4 (ratio to the survey 0.99 at the median pair): routing from every TAZ
centroid and weighting by population × employment adds 0.3 min to the point-to-point skim —
the representative points stand for their areas well for the car.

**Transit, TAZ level.** 10,997 stops, 1,410 line-directions, 46,263 line nodes, 44,726 ride
edges (97 % at step 30's observed times), 263,778 edges; a median TAZ reaches 32 stops within
a kilometre, 92 TAZs only their nearest stop. 608,400 of 609,180 pairs reachable (5,903 by
walking alone); GC median 150 generalized minutes over the whole area, **92 between the
corridor TAZs** (IVT 33, walk 15, wait 10, 0.72 transfers, 44 % direct, 54 % of paths riding
the Metronit). Aggregated to the areas: **45.2 generalized minutes on the trunk pairs**
(IVT 14.7, walk 9.6, wait 5.1, 0.1 transfers, 86 % of the TAZ pairs direct) against step 31's
**29.8** (cheaper of bus and Metronit, pooled wait) and **34.9** (best-line wait) — a median
pair ratio of 1.5 on the trunk and 1.45 over all 600 pairs (1.36 against the best-line
variant). The in-vehicle time is *lower* than step 31's (14.7 vs 16.1: the best path picks
the fastest line) and transfers are the same; the gap is **walk (9.6 vs 5.1 min, weighted
twice) and wait (5.1 vs 1.7 pooled / 3.8 best line)**. Both are the aggregation bias the
hand-over's C3 anticipated, seen from the other side: the area skim pairs the *nearest*
stop's walk with the *best* line's running time and the *pooled* headway of every direct
line, a combination no single traveller from a given TAZ can take — at TAZ level the line
that gives the direct ride is reached by a longer walk and waited for at its own headway.
Per TAZ, the transit / car GC ratio to the other corridor TAZs is 3.0 at the median
(2.6–3.7 for the central 80 %), best in the Krayot and Nazareth centres (2.2–2.4) and worst
in the employment-only zones of the Haifa bay and the Kiryat Ata fringe (4.2–5.2).

**Outputs** (`Output/los/`).
- `car_los_taz.csv.gz` — every reachable TAZ pair: `car_ivt_min` (kerb to kerb with connectors), `car_km`, `car_gc_min` (+ 3 min terminal).
- `transit_los_taz.csv.gz` — every reachable TAZ pair: `transit_gc_min`, `ivt_min`, `walk_min`, `wait_min`, `transfers`, `boardings`, `brt_ivt_min`, `walk_only`.
- `los_taz_accessibility.csv` — per TAZ: median car and transit GC to the V2-area TAZs, transit reach and direct share, the transit / car ratio.
- `{car_ivt,car_gc,car_km,transit_gc,transit_ivt,transit_walk,transit_wait,transit_transfers,transit_brt_ivt}_area_v2.csv`,
  `transit_direct_share_area_v2.csv`, `taz_pairs_reachable_area_v2.csv` — the 25 × 25 aggregates.
- `los_area_v2_pairs.csv` (600 pairs, this step beside step 31 and step 37), `los_area_v2_vs_step31.csv` (the summary table above).
- Figures `los_area_v2_taz_agg_vs_step31.png`, `los_taz_gc_maps.png`.

**Limits.** The transit skim is a single best path (its own line's headway, no pooling of
parallel lines — the chain's default pools them, which is why it is compared with the
`best_line` variant too); scheduled services of one Tuesday, 06:00–09:00 departures, with
step 30's observed running times on the segments it covered; walking on the straight line
× 1.3 (no walking network — task C2); access from the polygon centroid, which for a large
TAZ is a poor stand-in for where people live. The car skim carries step 37's assumptions
(30 km/h on the uncovered local segments, 20 km/h connectors, 3-minute terminal, speeds that
per caveat 24 are in practice all-day). Neither level carries money. The TAZ aggregates
weight pairs by population × employment, not by observed trips, so they are what a
population-wide traveller faces, not what the 2022 trip pattern experienced.